# DL300 Exercises -- CNNs: Convolution, Pooling, and Transfer Learning

Work through these exercises after completing the DL300 module. Try each exercise before looking at the solutions at the bottom.

**Topics covered:** Convolution output size calculation, pooling operations, transfer learning setup, data augmentation pitfalls, debugging normalization mismatches and augmentation on test data.

---
## Exercise 1: Convolution Output Size Calculation

For each scenario, compute the output spatial dimensions using the formula:

$$H_{out} = \left\lfloor\frac{H_{in} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}}\right\rfloor + 1$$

| Scenario | Input Size | Kernel | Stride | Padding | Output Size? |
|----------|-----------|--------|--------|---------|----------|
| A | 32x32 | 3x3 | 1 | 0 | ? |
| B | 32x32 | 3x3 | 1 | 1 | ? |
| C | 32x32 | 5x5 | 2 | 2 | ? |
| D | 224x224 | 7x7 | 2 | 3 | ? |
| E | 28x28 | 3x3 | 1 | 1, dilation=2 | ? |

**Verify your answers with PyTorch.** For dilation, the formula becomes:

$$H_{out} = \left\lfloor\frac{H_{in} + 2 \times \text{padding} - \text{dilation} \times (\text{kernel\_size} - 1) - 1}{\text{stride}}\right\rfloor + 1$$

In [ ]:
import torch
import torch.nn as nn

# Write your manual calculations here, then verify with PyTorch

# Scenario A: 32x32, kernel=3, stride=1, padding=0
# Manual: H_out = (32 + 0 - 3) / 1 + 1 = ?
conv_a = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0)
out_a = conv_a(torch.randn(1, 1, 32, 32))
print(f"A: {out_a.shape}")

# Scenario B: 32x32, kernel=3, stride=1, padding=1
# Manual: ?
conv_b = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
out_b = conv_b(torch.randn(1, 1, 32, 32))
print(f"B: {out_b.shape}")

# Scenario C: 32x32, kernel=5, stride=2, padding=2
# Manual: ?
conv_c = nn.Conv2d(1, 1, kernel_size=5, stride=2, padding=2)
out_c = conv_c(torch.randn(1, 1, 32, 32))
print(f"C: {out_c.shape}")

# Scenario D: 224x224, kernel=7, stride=2, padding=3
# Manual: ?
conv_d = nn.Conv2d(1, 1, kernel_size=7, stride=2, padding=3)
out_d = conv_d(torch.randn(1, 1, 224, 224))
print(f"D: {out_d.shape}")

# Scenario E: 28x28, kernel=3, stride=1, padding=1, dilation=2
# Manual: ?
conv_e = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, dilation=2)
out_e = conv_e(torch.randn(1, 1, 28, 28))
print(f"E: {out_e.shape}")

---
## Exercise 2: Pooling Operations

1. Given a 4x4 feature map, manually compute the result of:
   - MaxPool2d with kernel_size=2, stride=2
   - AvgPool2d with kernel_size=2, stride=2

2. What is the output size of `AdaptiveAvgPool2d(1)` applied to ANY spatial input? Why is this useful?

3. When would you prefer max pooling over average pooling?

In [ ]:
import torch
import torch.nn as nn

# 4x4 feature map
feature_map = torch.tensor([
    [1.0, 3.0, 2.0, 4.0],
    [5.0, 6.0, 1.0, 2.0],
    [3.0, 2.0, 7.0, 8.0],
    [4.0, 1.0, 5.0, 6.0],
]).unsqueeze(0).unsqueeze(0)  # (1, 1, 4, 4)

print(f"Input:\n{feature_map.squeeze()}\n")

# MaxPool2d
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
# TODO: Compute manually first, then verify
# Top-left 2x2 block: max(1,3,5,6) = ?
# Top-right 2x2 block: max(2,4,1,2) = ?
# Bottom-left 2x2 block: max(3,2,4,1) = ?
# Bottom-right 2x2 block: max(7,8,5,6) = ?
print(f"MaxPool result:\n{max_pool(feature_map).squeeze()}\n")

# AvgPool2d
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
# TODO: Compute manually
print(f"AvgPool result:\n{avg_pool(feature_map).squeeze()}\n")

# AdaptiveAvgPool2d(1)
adaptive = nn.AdaptiveAvgPool2d(1)
print(f"AdaptiveAvgPool(1) result: {adaptive(feature_map).squeeze()}")
# Works with any input size:
print(f"On 7x7 input: {adaptive(torch.randn(1,1,7,7)).shape}")
print(f"On 13x13 input: {adaptive(torch.randn(1,1,13,13)).shape}")

---
## Exercise 3: Transfer Learning Setup

You want to use a pretrained ResNet18 for a **binary classification** task (cats vs dogs) with 2,000 training images.

1. Load the pretrained model and freeze all parameters
2. Replace the final classification head for 2 classes
3. Count trainable vs frozen parameters
4. What optimizer and learning rate would you use?

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# TODO: Load pretrained ResNet18
# model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# TODO: Freeze all parameters
# for param in model.parameters():
#     param.requires_grad = False

# TODO: Replace model.fc with a new head for 2 classes
# in_features = model.fc.in_features
# model.fc = nn.Linear(in_features, 2)

# TODO: Count parameters
# total = sum(p.numel() for p in model.parameters())
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Total: {total:,} | Trainable: {trainable:,} | Frozen: {total-trainable:,}")

---
## Exercise 4: Data Augmentation -- Correct vs Incorrect

Review the following augmentation pipelines and identify which ones are **correct** and which have **bugs**.

### Pipeline A
```python
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),   # <--- Is this correct for test?
    transforms.RandomRotation(15),        # <--- Is this correct for test?
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
```

### Pipeline B
```python
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # <--- correct?
])
```

### Pipeline C
```python
# For a medical image classification task
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.5, contrast=0.5),
    transforms.ToTensor(),
])
```

For each pipeline, state if it is correct, and if not, explain the issue.

**Your answers:**

---
## Exercise 5: Debugging -- Normalization Mismatch

The code below trains a model using one set of normalization statistics but evaluates with different statistics. This is a common bug when using pretrained models.

1. Identify the bug
2. Explain why this causes poor test performance
3. Fix the code

In [ ]:
from torchvision import transforms

# Training: using ImageNet normalization (correct for pretrained models)
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                         std=[0.229, 0.224, 0.225]),
])

# Testing: OOPS -- using different normalization!
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],        # WRONG stats!
                         std=[0.5, 0.5, 0.5]),         # Different from training!
])

# TODO: Explain the bug and write the corrected test_transform

---
## Exercise 6: Debugging -- Augmentation on Test Data

A student reports that their model's test accuracy **changes every time they run evaluation**, even though no training is happening.

Look at their code and find the bug.

In [ ]:
from torchvision import transforms

# The student used the SAME transform for both train and test
shared_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),           # Random! Non-deterministic!
    transforms.RandomHorizontalFlip(),     # Random! Non-deterministic!
    transforms.ColorJitter(brightness=0.2),# Random! Non-deterministic!
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Used for both:
# train_dataset = ImageFolder("train/", transform=shared_transform)
# test_dataset = ImageFolder("test/", transform=shared_transform)  # <-- BUG

# TODO:
# 1. Explain why test accuracy varies between runs
# 2. Write the correct test_transform
# 3. What deterministic alternatives replace RandomCrop and RandomHorizontalFlip?

---
## Exercise 7: CNN Architecture Analysis

Trace through the following CNN and compute:
1. The output spatial size after each layer
2. The total number of parameters in each layer
3. The total number of parameters in the model

Input: (batch, 3, 32, 32)

In [ ]:
import torch
import torch.nn as nn

class AnalyzeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)    # TODO: params?
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # TODO: params?
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)   # TODO: params?
        self.fc1 = nn.Linear(64 * 4 * 4, 128)                     # TODO: params?
        self.fc2 = nn.Linear(128, 10)                              # TODO: params?

    def forward(self, x):
        # TODO: trace shapes through the network
        x = self.pool(torch.relu(self.conv1(x)))  # Shape after: ?
        x = self.pool(torch.relu(self.conv2(x)))  # Shape after: ?
        x = self.pool(torch.relu(self.conv3(x)))  # Shape after: ?
        x = x.view(x.size(0), -1)                 # Shape after: ?
        x = torch.relu(self.fc1(x))               # Shape after: ?
        x = self.fc2(x)                            # Shape after: ?
        return x

model = AnalyzeCNN()
x = torch.randn(1, 3, 32, 32)
output = model(x)
print(f"Final output shape: {output.shape}")

# Count parameters per layer
for name, param in model.named_parameters():
    print(f"{name:20s}: {param.shape} = {param.numel():,} params")

total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total:,}")

---
---
# Solutions

Expand each section below to see the solution.

<details>
<summary><b>Solution 1: Convolution Output Size Calculation</b></summary>

Using the formula: `H_out = floor((H_in + 2*padding - kernel) / stride) + 1`

| Scenario | Calculation | Output |
|----------|-------------|--------|
| A | (32 + 0 - 3) / 1 + 1 = 30 | **30x30** |
| B | (32 + 2 - 3) / 1 + 1 = 32 | **32x32** (same padding) |
| C | (32 + 4 - 5) / 2 + 1 = 16 | **16x16** (halves spatial dims) |
| D | (224 + 6 - 7) / 2 + 1 = 112 | **112x112** (ResNet first conv) |
| E | With dilation=2: effective kernel = 2*(3-1)+1 = 5. (28 + 2 - 5) / 1 + 1 = 26 | **26x26** |

**Key insights:**
- `padding=1, kernel=3, stride=1` preserves spatial dimensions ("same" padding)
- `stride=2` halves spatial dimensions (common for downsampling)
- Dilation increases the effective receptive field without adding parameters
</details>

<details>
<summary><b>Solution 2: Pooling Operations</b></summary>

**MaxPool2d(2,2) on the 4x4 input:**

```
Input:          MaxPool Result:
[1, 3 | 2, 4]    [6, 4]
[5, 6 | 1, 2]    [4, 8]
------+------
[3, 2 | 7, 8]
[4, 1 | 5, 6]
```

**AvgPool2d(2,2) on the 4x4 input:**

```
Input:          AvgPool Result:
[1, 3 | 2, 4]    [3.75, 2.25]
[5, 6 | 1, 2]    [2.50, 6.50]
------+------
[3, 2 | 7, 8]
[4, 1 | 5, 6]
```

**AdaptiveAvgPool2d(1)** always outputs a 1x1 spatial tensor (global average), regardless of input size. This is useful because it makes the network accept any input resolution -- there is no need to compute the exact flattened size before the fully connected layers.

**Max vs Average pooling:** Max pooling is preferred when you want to detect the presence of a feature (it keeps the strongest activation). Average pooling is preferred for global pooling before classification (it considers all spatial locations equally).
</details>

<details>
<summary><b>Solution 3: Transfer Learning Setup</b></summary>

```python
import torch
import torch.nn as nn
from torchvision import models

# 1. Load pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# 3. Replace classification head
in_features = model.fc.in_features  # 512
model.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2),
)

# 4. Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total:,} | Trainable: {trainable:,} | Frozen: {total-trainable:,}")
# Total: ~11.2M | Trainable: ~131K | Frozen: ~11.1M

# 5. Optimizer: Adam with lr=1e-3 for the new head
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
```

With only 2,000 training images, training only the head (131K params) is much safer than fine-tuning the full network (11M params), which would severely overfit.
</details>

<details>
<summary><b>Solution 4: Data Augmentation -- Correct vs Incorrect</b></summary>

**Pipeline A: BUG -- Augmentation applied to test data**
- `RandomHorizontalFlip` and `RandomRotation` should NOT be in the test transform. Test/validation data should be processed deterministically.
- **Fix:** Remove random augmentations from test_transform:
```python
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
```

**Pipeline B: POTENTIALLY WRONG -- Normalization mismatch**
- Using `mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]` is valid for training from scratch (maps [0,1] to [-1,1]), but if using a pretrained ImageNet model, you MUST use ImageNet statistics `mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]`.
- If training from scratch, this is acceptable.

**Pipeline C: DOMAIN-SPECIFIC CONCERN**
- For medical images, `RandomHorizontalFlip` and `RandomVerticalFlip` may be appropriate (e.g., pathology slides have no canonical orientation).
- However, `ColorJitter` can be problematic for medical images where color carries diagnostic information (e.g., staining intensity in histopathology). Use with caution and domain expertise.
- Also missing: `ToTensor()` normalization -- medical images often need careful normalization.
</details>

<details>
<summary><b>Solution 5: Debugging -- Normalization Mismatch</b></summary>

**Bug:** The training data is normalized with ImageNet statistics `(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`, but the test data uses different statistics `(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])`. This means the model sees test data in a different distribution than what it was trained on.

**Why this hurts:** The pretrained model's batch normalization layers and learned weights expect data normalized with ImageNet stats. Different normalization shifts and scales the activations, causing incorrect predictions.

**Fix:** Use the SAME normalization statistics for both training and testing:
```python
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # SAME stats
])
```
</details>

<details>
<summary><b>Solution 6: Debugging -- Augmentation on Test Data</b></summary>

**Bug:** The test dataset uses the same transform as training, which includes random augmentations (`RandomCrop`, `RandomHorizontalFlip`, `ColorJitter`). Each evaluation run applies different random transformations, producing different predictions.

**Why test accuracy varies:** `RandomCrop(224)` crops a random 224x224 patch from the 256x256 image. Different crops show different parts of the image, leading to different predictions.

**Fix:** Use deterministic transforms for testing:
```python
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),           # Deterministic: always takes center
    # NO RandomHorizontalFlip
    # NO ColorJitter
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
```

**Deterministic alternatives:**
- `RandomCrop(224)` -> `CenterCrop(224)` (deterministic center crop)
- `RandomHorizontalFlip()` -> removed entirely (or use test-time augmentation intentionally)
- `ColorJitter()` -> removed entirely
</details>

<details>
<summary><b>Solution 7: CNN Architecture Analysis</b></summary>

**Layer-by-layer trace (input: (1, 3, 32, 32)):**

```
Layer         | Output Shape      | Parameters
--------------|-------------------|-----------
conv1 (3x3)   | (1, 16, 32, 32)  | 3*16*3*3 + 16 = 448
pool (2x2)    | (1, 16, 16, 16)  | 0
conv2 (3x3)   | (1, 32, 16, 16)  | 16*32*3*3 + 32 = 4,640
pool (2x2)    | (1, 32, 8, 8)    | 0
conv3 (3x3)   | (1, 64, 8, 8)    | 32*64*3*3 + 64 = 18,496
pool (2x2)    | (1, 64, 4, 4)    | 0
flatten        | (1, 1024)        | 0
fc1            | (1, 128)         | 1024*128 + 128 = 131,200
fc2            | (1, 10)          | 128*10 + 10 = 1,290
```

**Total parameters:** 448 + 4,640 + 18,496 + 131,200 + 1,290 = **156,074**

**Key observation:** The `fc1` layer has 131,200 parameters -- **84%** of all parameters! This is typical of CNNs where the transition from convolutional to fully connected layers creates a parameter explosion. Modern architectures use global average pooling to avoid this.

**Conv parameter formula:** `in_channels * out_channels * kernel_h * kernel_w + out_channels (bias)`
</details>